# VoyageAI - Hotel Agent

This notebook creates the Hotel Agent for VoyageAI.

The Hotel Agent is responsible for:
1. Reading the selected destination from the Destination Agent
2. Retrieving stay-related information from ChromaDB
3. Suggesting suitable stay types
4. Returning structured hotel/stay recommendations

This version does not use live hotel APIs.
It uses RAG knowledge from our travel documents.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import json

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

e:\Agentic AI\VoyageAI Backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

print("Environment variables loaded.")

Environment variables loaded.


In [3]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "agents":
    PROJECT_ROOT = CURRENT_DIR.parent.parent
elif CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

CHROMA_DIR = PROJECT_ROOT / "chroma_db"

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Chroma DB folder:", CHROMA_DIR)

if not CHROMA_DIR.exists():
    raise FileNotFoundError("Chroma DB folder not found. Run 01_rag_ingestion.ipynb first.")

Current directory: e:\Agentic AI\VoyageAI Backend\notebooks\agents
Project root: e:\Agentic AI\VoyageAI Backend
Chroma DB folder: e:\Agentic AI\VoyageAI Backend\chroma_db


In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10603.38it/s]


Embedding model loaded successfully.


In [5]:
vector_store = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embedding_model,
    collection_name="voyageai_travel_knowledge"
)

print("ChromaDB loaded successfully.")
print("Total vectors:", vector_store._collection.count())

ChromaDB loaded successfully.
Total vectors: 45


In [6]:
DESTINATION_KEY_MAP = {
    "goa": "goa",
    "jaipur": "jaipur",
    "manali": "manali",
    "kerala": "kerala",
    "andaman": "andaman",
    "andaman and nicobar islands": "andaman",
    "udaipur": "udaipur",
    "rishikesh": "rishikesh",
    "varanasi": "varanasi",
    "ladakh": "ladakh",
    "kashmir": "kashmir",
    "darjeeling": "darjeeling",
    "pondicherry": "pondicherry",
    "puducherry": "pondicherry",
    "agra": "agra",
    "delhi": "delhi",
    "mumbai": "mumbai"
}


def normalize_destination_to_key(destination_name: str):
    if not destination_name:
        return None

    destination_name = destination_name.lower().strip()

    for name, key in DESTINATION_KEY_MAP.items():
        if name in destination_name:
            return key

    return destination_name.replace(" ", "_")

In [7]:
def get_hotel_context(user_query: str, destination_result: dict, k: int = 4):
    destination = destination_result.get("recommended_destination")
    city_key = normalize_destination_to_key(destination)

    retrieval_query = f"""
    Destination: {destination}
    User request: {user_query}
    Need hotel stay suggestions, budget stay, comfort stay, luxury stay, best areas, accommodation tips.
    """

    try:
        docs = vector_store.similarity_search(
            retrieval_query,
            k=k,
            filter={"city": city_key}
        )

        if not docs:
            docs = vector_store.similarity_search(retrieval_query, k=k)

    except Exception as e:
        print("Filter search failed. Running normal similarity search.")
        print("Error:", e)
        docs = vector_store.similarity_search(retrieval_query, k=k)

    context_parts = []

    for i, doc in enumerate(docs, start=1):
        city = doc.metadata.get("city")
        source = doc.metadata.get("source")

        context = f"""
Context {i}
City: {city}
Source: {source}
Content:
{doc.page_content}
"""
        context_parts.append(context.strip())

    return "\n\n".join(context_parts)

In [ ]:
user_query = "I want beaches, seafood, nightlife and a relaxed vacation. My budget is low."

destination_result = {
    "agent_name": "Destination Agent",
    "recommended_destination": "Goa",
    "reason": "Goa matches the user's preference for beaches, seafood, nightlife and relaxed coastal travel.",
    "suitable_for": ["beach lovers", "food lovers", "nightlife travelers"],
    "suggested_duration": "3 to 5 days",
    "best_time_to_visit": "November to February",
    "confidence": "high"
}

hotel_context = get_hotel_context(user_query, destination_result)

print(hotel_context)

Context 1
City: goa
Source: goa.txt
Content:
Destination: Goa
State/Region: Goa
Destination Type: Beach, nightlife, seafood, coastal vacation, friends trip, couples trip

Overview:
Goa is one of India's most popular beach destinations. It is known for beaches, nightlife, seafood, Portuguese heritage, forts, churches, markets, and relaxed coastal culture.

Best For:
- Beach lovers
- Seafood lovers
- Friends groups
- Couples
- Nightlife travelers
- Relaxed vacation seekers

Context 2
City: goa
Source: goa.txt
Content:
Top Attractions:
- Baga Beach
- Calangute Beach
- Anjuna Beach
- Vagator Beach
- Fort Aguada
- Chapora Fort
- Basilica of Bom Jesus
- Panjim
- Dudhsagar Falls

Food Recommendations:
- Goan fish curry
- Prawn balchao
- Pork vindaloo
- Xacuti
- Bebinca
- Seafood thali

Stay Suggestions:
- Budget: hostels near Anjuna, Vagator, or Baga
- Comfort: boutique hotels near Candolim or Panjim
- Luxury: beach resorts in South Goa

Context 3
City: goa
Source: goa.txt
Content:
Best Time 

In [9]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)

print("Groq LLM loaded successfully.")

Groq LLM loaded successfully.


In [10]:
hotel_agent_prompt = ChatPromptTemplate.from_template("""
You are the Hotel Agent of VoyageAI.

Your task is to suggest the best stay type for the user based on:
1. User travel request
2. Destination Agent output
3. Retrieved travel knowledge

User Travel Request:
{user_query}

Destination Agent Output:
{destination_result}

Retrieved Travel Knowledge:
{context}

Return your answer strictly in valid JSON format.

JSON structure:
{{
  "agent_name": "Hotel Agent",
  "destination": "destination name",
  "budget_preference": "budget/comfort/luxury/unknown",
  "recommended_stay_type": "short recommendation",
  "best_areas": ["area 1", "area 2", "area 3"],
  "stay_options": {{
    "budget": "budget stay suggestion",
    "comfort": "comfort stay suggestion",
    "luxury": "luxury stay suggestion"
  }},
  "reason": "why this stay recommendation matches the user",
  "hotel_booking_tip": "practical booking tip",
  "confidence": "high/medium/low"
}}

Rules:
- Use only the retrieved travel knowledge.
- Do not invent exact hotel names.
- Do not invent exact prices.
- Suggest stay categories and areas only.
- If the user's budget is not mentioned, set budget_preference to "unknown".
- Do not add markdown.
- Do not wrap JSON in ```json.
""")

In [11]:
hotel_agent_chain = hotel_agent_prompt | llm | StrOutputParser()

print("Hotel Agent chain created successfully.")

Hotel Agent chain created successfully.


In [12]:
def parse_json_response(response: str):
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        try:
            start = response.find("{")
            end = response.rfind("}") + 1
            json_str = response[start:end]
            return json.loads(json_str)
        except Exception:
            print("Failed to parse JSON. Raw response:")
            print(response)
            return {
                "error": "Failed to parse JSON",
                "raw_response": response
            }

In [21]:
def get_available_destinations():
    collection_data = vector_store._collection.get(include=["metadatas"])

    destinations = set()

    for metadata in collection_data["metadatas"]:
        city = metadata.get("city")
        if city:
            destinations.add(city.lower().strip())

    return sorted(list(destinations))


available_destinations = get_available_destinations()

print("Available destinations in ChromaDB:")
print(available_destinations)

Available destinations in ChromaDB:
['agra', 'andaman', 'darjeeling', 'delhi', 'goa', 'jaipur', 'kashmir', 'kerala', 'ladakh', 'manali', 'mumbai', 'pondicherry', 'rishikesh', 'udaipur', 'varanasi']


In [22]:
def normalize_text(text: str):
    if not text:
        return ""

    return text.lower().strip()


def is_destination_in_kb(destination_name: str):
    destination_name = normalize_text(destination_name)

    if not destination_name:
        return False

    for destination in available_destinations:
        if destination == destination_name:
            return True

        if destination in destination_name:
            return True

        if destination_name in destination:
            return True

    return False

In [23]:
def run_hotel_agent(user_query: str, destination_result: dict, k: int = 4):
    destination_status = destination_result.get("status")
    destination = destination_result.get("recommended_destination")

    if destination_status != "success":
        return {
            "agent_name": "Hotel Agent",
            "status": "skipped",
            "destination": None,
            "message": "Hotel Agent skipped because Destination Agent did not return a valid destination.",
            "reason": destination_result.get("reason"),
            "confidence": "low"
        }

    if not destination:
        return {
            "agent_name": "Hotel Agent",
            "status": "no_destination_found",
            "destination": None,
            "message": "Hotel Agent cannot run because no destination was provided.",
            "confidence": "low"
        }

    if not is_destination_in_kb(destination):
        return {
            "agent_name": "Hotel Agent",
            "status": "out_of_knowledge_base",
            "destination": destination,
            "message": f"{destination} is not available in the current VoyageAI knowledge base.",
            "available_destinations": available_destinations,
            "confidence": "low"
        }

    context = get_hotel_context(
        user_query=user_query,
        destination_result=destination_result,
        k=k
    )

    response = hotel_agent_chain.invoke({
        "user_query": user_query,
        "destination_result": json.dumps(destination_result, indent=2),
        "context": context
    })

    parsed_response = parse_json_response(response)

    parsed_response["status"] = parsed_response.get("status", "success")

    return parsed_response

In [ ]:
user_query = "I want a romantic luxury trip with lakes and palaces"

destination_agent_result = {
  "agent_name": "Destination Agent",
  "status": "success",
  "recommended_destination": "udaipur",
  "reason": "palaces and lakes",
  "suitable_for": [
    "romantic",
    "luxury"
  ],
  "suggested_duration": "4-5 days",
  "best_time_to_visit": "October to March",
  "key_attractions": [
    "City Palace",
    "Lake Pichola",
    "Jag Mandir"
  ],
  "confidence": "high"
}

hotel_result = run_hotel_agent(user_query, destination_agent_result)

print(json.dumps(hotel_result, indent=2))

{
  "agent_name": "Hotel Agent",
  "destination": "Udaipur",
  "budget_preference": "unknown",
  "recommended_stay_type": "lake-view boutique hotels or palace hotels and lake resorts",
  "best_areas": [
    "old city",
    "near Lake Pichola"
  ],
  "stay_options": {
    "budget": "guesthouses near old city",
    "comfort": "lake-view boutique hotels",
    "luxury": "palace hotels and lake resorts"
  },
  "reason": "Udaipur is known for its lake views and royal palaces, making lake-view boutique hotels or palace hotels and lake resorts a suitable choice.",
  "hotel_booking_tip": "Choose a lake-view stay for a better experience and book in advance to ensure availability.",
  "confidence": "high",
  "status": "success"
}
